# NB-01 · Hybrid Preprocessing
### Bubble Flow Analysis System — Stage 1

**Roadmap reference:** Stage 1 — Hybrid Preprocessing (Section 2, NB-01 in Section 7)  
**Gate in:** validated `config.json` produced by NB-00  
**Gate out:** preprocessed frames per chunk (PNG / NPY) + per-frame quality log  
**Acceptance criterion:** visual before/after inspection; intensity histogram per chunk confirms signal improvement  

---
### Video characterisation (VID_20260413_124627.mp4)

| Property | Value |
|---|---|
| Resolution | 2160 × 3840 px (portrait 4K) |
| FPS | 44.64 |
| Duration | 8.58 s / 383 frames |
| Codec | H.264 |
| Illumination | Warm backlight / IR — bubbles appear **bright** on a **dark** background |
| Column orientation | Vertical — bubbles rise from sparger at y≈3778 toward top |
| Gas nozzle | x≈1080 px (frame centre) |

**Pipeline order (roadmap § 1.2):**  
`crop → background subtraction (MOG2) → contrast normalisation → bilateral denoise → optional adaptive threshold → quality log`  
No heavy CNN is used here — deterministic OpenCV operations first.

In [ ]:
import sys, subprocess

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        print('Drive mounted at /content/drive')
    except Exception as _e:
        if 'already contain files' in str(_e) or 'symlink' in str(_e):
            print('Stale mount — clearing and remounting...')
            subprocess.run(['umount', '/content/drive'], capture_output=True)
            subprocess.run(['rm', '-rf', '/content/drive'], capture_output=True)
            drive.mount('/content/drive', force_remount=False)
            print('Drive remounted successfully.')
        else:
            raise
else:
    print('Not in Colab — skipping Drive mount.')


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

## Cell 0B — Dependency Check

In [ ]:
import sys, subprocess, importlib.util

# Check for missing packages and install only what is needed.
# This avoids redundant installs on every session restart.
REQUIRED = [
    ('opencv-python-headless', 'cv2'),
    ('pyarrow',                'pyarrow'),
    ('tqdm',                   'tqdm'),
]
missing = [pkg for pkg, mod in REQUIRED if importlib.util.find_spec(mod) is None]

if missing:
    print('Installing missing packages:', missing)
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + missing,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError('Dependency installation failed. See output above.')
    print('Packages installed. Restart runtime if imports still fail.')
else:
    print('All dependencies present.')


## Cell 0C — Path Configuration

In [ ]:
from pathlib import Path

# ── EDIT THIS LINE ────────────────────────────────────────────────────────────
DRIVE_PROJECT_PATH = 'SIGNAL_NN_2026/HIDRO_2026'   # folder inside MyDrive
# ─────────────────────────────────────────────────────────────────────────────

if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_PROJECT_PATH
else:
    DRIVE_ROOT = Path.home() / 'BubbleFlow'

print('DRIVE_ROOT   :', DRIVE_ROOT)
print('Drive exists :', DRIVE_ROOT.exists())


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

matplotlib.rcParams['figure.dpi'] = 110
warnings.filterwarnings('ignore')

print('NB-01  ·  Preprocessing  ·  imports OK')

---
## Cell 1 — Configuration
Edit only this cell. All parameters for this notebook are set here.

| Cell | Creates |
|---|---|
| 0A | Drive mount |
| 0B | Installed packages |
| 0C | `DRIVE_ROOT` |
| **1** | **All path and algorithm parameters** |
| 2 | `load_config()`, `ROI`, `FPS`, `V_FRAMES`, video metadata |
| 3 | `CHUNKS` chunk plan |
| 4 | All processing functions |
| 5 | Frame extraction check |
| 6 | Before/after visualisation |
| 7 | Batch processing (all chunks) |
| 8 | Quality log saved |
| 9 | Gate report |
| 10 | Summary Markdown report |

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
VIDEO_PATH   : str = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/run_001_Q10/VID_20260413_124627.mp4"
CONFIG_PATH  : str = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/config.json"
OUTPUT_DIR   : str = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/campaigns/outputs/preproc/"

# ── CAMPAIGN LABEL ────────────────────────────────────────────────────────────
# Must match RUN_LABEL in NB-02 through NB-06 and MANUAL_FILES key in NB-06.
RUN_LABEL    : str = 'run_001_Q10'

# ── CHUNK STRATEGY ────────────────────────────────────────────────────────────
# 383 frames @ 44.64 fps → 8.58 s
# 96-frame chunks with 32-frame overlap → 6 chunks covering full video
CHUNK_SIZE   : int   = 96     # frames per chunk
CHUNK_OVERLAP: int   = 32     # overlap between consecutive chunks

# ── OUTPUT FORMAT ─────────────────────────────────────────────────────────────
# 'png' — lossless, recommended for segmentation input
# 'npy' — raw float32, faster to load in Python
OUTPUT_FORMAT: str   = 'png'

# ── FALLBACK ROI (used if config.json is missing or has full-frame ROI) ───────
# Measured from portrait video frame (2160 x 3840 px).
# Column body: x=[900,1260], y=[150,3720] — tight crop, excludes sparger hardware.
FALLBACK_ROI           : list = [0, 450, 2160, 3400]
FALLBACK_REFERENCE_LINES: list = [1187, 1925, 2662]

# ── BACKGROUND SUBTRACTION ────────────────────────────────────────────────────
MOG2_HISTORY         : int   = 100
MOG2_VAR_THRESHOLD   : float = 25.0
MOG2_DETECT_SHADOWS  : bool  = False   # shadows irrelevant in backlit setup

# ── CONTRAST NORMALISATION ────────────────────────────────────────────────────
CLAHE_CLIP_LIMIT : float = 2.5
CLAHE_TILE_GRID  : tuple = (8, 8)

# ── DENOISING ─────────────────────────────────────────────────────────────────
BILATERAL_D       : int   = 7
BILATERAL_SIGMA_C : float = 50.0
BILATERAL_SIGMA_S : float = 50.0

# ── ADAPTIVE THRESHOLD (optional) ────────────────────────────────────────────
USE_ADAPTIVE_THRESHOLD : bool  = False   # enable if MOG2 misses faint bubbles
ADAPT_BLOCK_SIZE       : int   = 31
ADAPT_C                : int   = -5      # negative = keep bright regions

# ── QUALITY GATE ──────────────────────────────────────────────────────────────
SNR_FLOOR_DB          : float = 20.0    # Otsu-based SNR floor (dB)
MAX_BAD_FRAME_FRACTION: float = 0.10    # gate fails if > 10% frames flagged
GATE_PCT_THRESHOLD    : float = (1.0 - MAX_BAD_FRAME_FRACTION) * 100.0

# ── INFERENCE DOWNSCALE ───────────────────────────────────────────────────────
INFERENCE_SCALE : float = 1.0    # 0.5 → 180×1785 px per frame

# ── INSPECTION ────────────────────────────────────────────────────────────────
INSPECT_FRAME_IDX : int = 100     # frame index for visual checks

# ── PREPARE OUTPUT ────────────────────────────────────────────────────────────
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

print(f'Video          : {VIDEO_PATH}')
print(f'Config         : {CONFIG_PATH}')
print(f'Output dir     : {OUT.resolve()}')
print(f'Chunk size     : {CHUNK_SIZE} frames  overlap={CHUNK_OVERLAP}')
print(f'Inference scale: {INFERENCE_SCALE}x')
print(f'Gate threshold : >= {GATE_PCT_THRESHOLD:.1f}% OK frames')
print(f'Fallback ROI   : {FALLBACK_ROI}')


---
## Cell 2 — Load config.json and Video Metadata

In [ ]:
import json as _json_pf
_cfg_p = Path(CONFIG_PATH)
if not _cfg_p.exists():
    raise FileNotFoundError(
        f'config.json not found: {_cfg_p.resolve()}\n'
        'Run NB-00 fully before running NB-01.'
    )
_cfg_pre = _json_pf.load(open(_cfg_p))
print('=' * 55)
print('NB-01 CONFIG PRE-FLIGHT CHECK')
print('=' * 55)
_expected = {
    'mm_per_pixel':       (0.20, 0.25,  'should be ~0.231'),
    'column_diameter_mm': (400,  600,   'should be 500.0'),
    'fps':                (40,   50,    'should be ~44.64'),
}
_preflight_ok = True
for _field, (_lo, _hi, _note) in _expected.items():
    _val = _cfg_pre.get(_field)
    _ok  = _val is not None and _lo <= float(_val) <= _hi
    _status = 'OK  ' if _ok else 'FAIL'
    print(f'  {_status}  {_field:<25} : {_val}  ({_note})')
    if not _ok:
        _preflight_ok = False
_roi = _cfg_pre.get('roi_coords', [])
_roi_ok = _roi == [0, 450, 2160, 3400]
_roi_status = 'OK  ' if _roi_ok else 'FAIL'
print(f'  {_roi_status}  roi_coords                : {_roi}')
if not _roi_ok:
    _preflight_ok = False
print('=' * 55)
if not _preflight_ok:
    raise RuntimeError(
        'config.json pre-flight FAILED.\n'
        'Re-run NB-00 from Cell 1 and ensure all values are correct.'
    )
print('config.json pre-flight PASSED — proceeding to load.')
print(f'Created at: {_cfg_pre.get("created_at", "unknown")}')
print()

In [ ]:
from pathlib import Path
chunks = sorted(Path(OUTPUT_DIR).glob('chunk_*'))
total = 0
for c in chunks:
    n = len(list(c.glob('frame_*.png')))
    print(f'  {c.name}: {n} frames')
    total += n
print(f'Total with overlap: {total}')

In [ ]:
# ── load_config() — calibration output from NB-00 ─────────────────────────
def load_config(path: str = 'config.json') -> dict:
    MANDATORY = [
        'mm_per_pixel', 'mm_per_pixel_uncertainty', 'fps',
        'roi_coords', 'column_diameter_mm', 'reference_lines_px',
        'schema_version', 'created_at',
    ]
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"config.json not found at '{p.resolve()}'.\n"
            "Run NB-00 first, or set FALLBACK_ROI in Cell 1."
        )
    with open(p, encoding='utf-8') as f:
        cfg_json = json.load(f)
    missing = [k for k in MANDATORY if k not in cfg_json]
    if missing:
        raise KeyError(f"config.json missing fields: {missing}")
    return cfg_json


def is_full_frame_roi(roi, width=2160, height=3840):
    return list(map(int, roi)) == [0, 0, int(width), int(height)]


try:
    calib_cfg = load_config(CONFIG_PATH)
    ROI = calib_cfg['roi_coords']
    FPS = calib_cfg['fps']
    MPP = calib_cfg['mm_per_pixel']
    REF_LINES = calib_cfg['reference_lines_px']
    CONFIG_AVAILABLE = True
    print(f"config.json loaded  (schema {calib_cfg['schema_version']}, created {calib_cfg['created_at']})")
except FileNotFoundError:
    print('[WARNING] config.json not found — using FALLBACK_ROI from Cell 1.')
    calib_cfg = {}
    ROI = FALLBACK_ROI
    FPS = 44.638
    MPP = None
    REF_LINES = FALLBACK_REFERENCE_LINES
    CONFIG_AVAILABLE = False

# Avoid accidental full-frame processing. Use focused ROI selected from the reference frame.
if is_full_frame_roi(ROI):
    print('[WARNING] config.json has full-frame ROI — replacing with FALLBACK_ROI from Cell 1.')
    ROI = FALLBACK_ROI
    REF_LINES = FALLBACK_REFERENCE_LINES

ROI = [int(v) for v in ROI]
REF_LINES = [int(v) for v in REF_LINES]

print(f"  mm_per_pixel : {MPP if MPP is not None else 'not available'}")
print(f"  ROI          : {ROI}")
print(f"  FPS          : {FPS:.4f}")
print(f"  Ref. lines   : {REF_LINES}")

video_path = Path('/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/run_001_Q10/VID_20260413_124627.mp4')
if not video_path.exists():
    raise FileNotFoundError(f"Video not found: {video_path.resolve()}")

cap = cv2.VideoCapture(str(video_path))
V_FPS = cap.get(cv2.CAP_PROP_FPS)
V_W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
V_H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
V_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"\nVideo metadata")
print(f"  File          : {video_path.name}")
print(f"  Resolution    : {V_W} x {V_H} px  ({'portrait' if V_H > V_W else 'landscape'})")
print(f"  FPS (header)  : {V_FPS:.4f}")
print(f"  Total frames  : {V_FRAMES}")
print(f"  Duration      : {V_FRAMES / V_FPS:.2f} s")

EFFECTIVE_FPS = FPS if CONFIG_AVAILABLE else V_FPS
print(f"  Effective FPS : {EFFECTIVE_FPS:.4f} (from {'config.json' if CONFIG_AVAILABLE else 'video header'})")

# Portrait orientation check
if V_H < V_W:
    print('[WARNING] Video appears to be landscape. Expected portrait (H > W).')
    print('  Proceeding — set FALLBACK_ROI in Cell 1 for correct geometry.')

---
## Cell 3 — ROI Guardrail
Confirms the ROI is not accidentally set to the full frame before batch processing.

In [ ]:
# Guard: confirm ROI is not full-frame before running batch processing.
# A full-frame ROI wastes compute and produces poor segmentation inputs.
print('ROI          :', ROI)
print('Reference lines:', REF_LINES)

if list(map(int, ROI)) == [0, 0, V_W, V_H]:
    raise ValueError(
        'ROI is the full frame [0, 0, ' + str(V_W) + ', ' + str(V_H) + '].\n'
        'Set FALLBACK_ROI in Cell 1 to the column body coordinates, e.g. [900, 150, 1260, 3720].\n'
        'Or run NB-00 to produce a calibrated config.json with the correct ROI.'
    )

if len(REF_LINES) < 3:
    raise ValueError(
        'reference_lines_px has ' + str(len(REF_LINES)) + ' entries — need at least 3.\n'
        'Set FALLBACK_REFERENCE_LINES in Cell 1 to three Y-pixel positions, e.g. [1043, 1935, 2828].'
    )

print('ROI guardrail: PASS')


---
## Cell 4 — Core Preprocessing Functions

In [ ]:
def build_chunk_plan(n_frames: int, chunk_size: int, overlap: int) -> list:
    """
    Build a list of (chunk_id, start_frame, end_frame) tuples.
    Chunks overlap by `overlap` frames for identity stitching (roadmap § 3).
    The last chunk is extended to cover all remaining frames.
    """
    step = chunk_size - overlap
    chunks = []
    cid = 0
    start = 0
    while start < n_frames:
        end = min(start + chunk_size, n_frames)
        chunks.append((cid, start, end))
        cid += 1
        start += step
        if end == n_frames:
            break
    return chunks


CHUNKS = build_chunk_plan(V_FRAMES, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Chunk plan  ({len(CHUNKS)} chunks, step={CHUNK_SIZE - CHUNK_OVERLAP} frames)")
print(f"{'Chunk':>5}  {'Start':>6}  {'End':>6}  {'N':>5}  {'t_start':>8}  {'t_end':>8}")
print("-" * 50)
for cid, s, e in CHUNKS:
    print(f"  {cid:3d}    {s:5d}   {e:5d}   {e-s:4d}   "
          f"{s/EFFECTIVE_FPS:6.3f}s   {e/EFFECTIVE_FPS:6.3f}s")

---
## Cell 5 — Processing Functions
All functions are defined here. Run once per session before Cell 6.

In [ ]:
# ── crop_to_roi() ─────────────────────────────────────────────────────────────

def crop_to_roi(
    frame: np.ndarray,
    roi: list,
    scale: float = 1.0,
) -> tuple:
    """
    Crop a frame to the ROI and optionally downscale for inference speed.

    Parameters
    ----------
    frame : BGR or grayscale image
    roi   : [x0, y0, x1, y1] in original pixel coordinates
    scale : downscale factor (1.0 = no scaling).
            Scale factors are preserved in the returned metadata so all
            physical outputs remain traceable to calibrated units.

    Returns
    -------
    (cropped_frame, scale_x, scale_y)
    scale_x, scale_y are the inverse scale factors (original_px / inference_px)
    to be multiplied back when converting inference coordinates to physical units.
    """
    x0, y0, x1, y1 = roi
    # Clamp to frame dimensions
    h, w = frame.shape[:2]
    x0 = max(0, x0); y0 = max(0, y0)
    x1 = min(w, x1); y1 = min(h, y1)

    cropped = frame[y0:y1, x0:x1]

    if scale != 1.0:
        new_w = max(1, int(cropped.shape[1] * scale))
        new_h = max(1, int(cropped.shape[0] * scale))
        cropped = cv2.resize(cropped, (new_w, new_h),
                             interpolation=cv2.INTER_AREA)

    scale_x = 1.0 / scale  # multiply inference coords by this → original coords
    scale_y = 1.0 / scale
    return cropped, scale_x, scale_y


print('crop_to_roi()  defined.')

In [ ]:
# ── background_subtraction_mog2() ────────────────────────────────────────────

def background_subtraction_mog2(
    frames_gray: list,
    history: int = 100,
    var_threshold: float = 25.0,
    detect_shadows: bool = False,
) -> tuple:
    """
    Apply MOG2 background subtraction to a list of grayscale frames.

    Bubbles appear BRIGHT on a DARK background (backlit / IR illumination).
    The background model learns the static dark field; bright moving bubbles
    produce high foreground probability masks.

    Parameters
    ----------
    frames_gray   : list of (H, W) uint8 grayscale arrays (one chunk)
    history       : number of frames in the MOG2 history
    var_threshold : Mahalanobis distance threshold (lower = more sensitive)
    detect_shadows: if True, shadows are marked at intensity 127

    Returns
    -------
    fg_masks : list of (H, W) uint8 binary masks  (255 = foreground / bubble)
    bg_model : final background image (H, W) uint8
    """
    fgbg = cv2.createBackgroundSubtractorMOG2(
        history=history,
        varThreshold=var_threshold,
        detectShadows=detect_shadows,
    )

    fg_masks = []
    for f in frames_gray:
        mask = fgbg.apply(f)
        # Convert shadow label (127) to foreground
        if detect_shadows:
            mask[mask == 127] = 255
        fg_masks.append(mask)

    bg_model = fgbg.getBackgroundImage()
    if bg_model is None:
        bg_model = np.zeros_like(frames_gray[0])

    return fg_masks, bg_model


print('background_subtraction_mog2()  defined.')

In [ ]:
# ── normalize_contrast() ──────────────────────────────────────────────────────

def normalize_contrast(
    frame_gray: np.ndarray,
    clip_limit: float = 2.5,
    tile_grid: tuple = (8, 8),
) -> np.ndarray:
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to a
    grayscale frame.  CLAHE preserves local contrast while preventing noise
    amplification in uniform regions (roadmap: 'local contrast normalisation').

    This step is particularly useful for the bubble-column video because
    the column has a bright sparger region at the bottom and a darker region
    at the top — CLAHE compensates for the vertical intensity gradient.

    Parameters
    ----------
    frame_gray : (H, W) uint8 grayscale image
    clip_limit : threshold for contrast limiting (higher = more contrast)
    tile_grid  : (rows, cols) of tiles for local histogram computation

    Returns
    -------
    (H, W) uint8 contrast-normalised image
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(frame_gray)


print('normalize_contrast()  defined.')

In [ ]:
# ── denoise_bilateral() ───────────────────────────────────────────────────────

def denoise_bilateral(
    frame_gray: np.ndarray,
    d: int   = 7,
    sigma_color: float = 50.0,
    sigma_space: float = 50.0,
) -> np.ndarray:
    """
    Apply bilateral filtering for edge-preserving smoothing.

    Bilateral filtering smooths noise while preserving the sharp boundaries
    of bubble edges — critical for accurate mask extraction in Stage 2.
    Unlike Gaussian blur, it does NOT blur across intensity edges.

    Parameters
    ----------
    frame_gray  : (H, W) uint8 grayscale image
    d           : filter neighbourhood diameter (px)
    sigma_color : filter sigma in the intensity domain
    sigma_space : filter sigma in the coordinate domain

    Returns
    -------
    (H, W) uint8 denoised image
    """
    return cv2.bilateralFilter(
        frame_gray.astype(np.uint8),
        d=d,
        sigmaColor=sigma_color,
        sigmaSpace=sigma_space,
    )


print('denoise_bilateral()  defined.')

In [ ]:
# ── adaptive_threshold() ─────────────────────────────────────────────────────

def adaptive_threshold(
    frame_gray: np.ndarray,
    block_size: int   = 31,
    C: int            = -5,
) -> np.ndarray:
    """
    Apply Gaussian adaptive thresholding as an optional fallback when MOG2
    foreground masks are weak (very low-contrast frames or transient regimes).

    Negative C keeps regions brighter than their local neighbourhood
    (i.e., bright bubbles on a dark background).

    Parameters
    ----------
    frame_gray : (H, W) uint8 normalised grayscale image
    block_size : neighbourhood size (must be odd)
    C          : constant subtracted from the mean (negative → keep bright)

    Returns
    -------
    (H, W) uint8 binary mask  (255 = bright foreground)
    """
    if block_size % 2 == 0:
        block_size += 1
    return cv2.adaptiveThreshold(
        frame_gray,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY,
        blockSize=block_size,
        C=C,
    )


print('adaptive_threshold()  defined.')

In [ ]:
# ── quality metrics for up6 ──────────────────────────────────────────────────

def blur_score_laplacian(frame_gray: np.ndarray) -> float:
    """Variance of Laplacian. Low values indicate blur or poor edge definition."""
    return float(cv2.Laplacian(frame_gray.astype(np.uint8), cv2.CV_64F).var())


def saturation_rate(frame_gray: np.ndarray, low: int = 2, high: int = 253) -> float:
    """Fraction of pixels clipped near black or white."""
    img = frame_gray.astype(np.uint8)
    return float(((img <= low) | (img >= high)).mean())


def local_contrast(frame_gray: np.ndarray, tile: int = 32) -> float:
    """Average local standard deviation over non-overlapping tiles."""
    img = frame_gray.astype(np.float32)
    h, w = img.shape[:2]
    vals = []
    for y in range(0, h - tile + 1, tile):
        for x in range(0, w - tile + 1, tile):
            vals.append(img[y:y+tile, x:x+tile].std())
    return float(np.mean(vals)) if vals else float(img.std())


def log_frame_quality(
    frame_orig: np.ndarray,
    frame_proc: np.ndarray,
    fg_mask: np.ndarray,
    frame_id: int,
    chunk_id: int,
    snr_floor: float = 22.0,
) -> dict:
    """
    Compute per-frame quality metrics for the quality log.

    adds blur, saturation and local contrast. These metrics help diagnose
    false detections caused by blur, clipped injector light, or weak bubble edges.
    """
    mean_raw = float(frame_orig.mean())
    std_raw = float(frame_orig.std())
    mean_proc = float(frame_proc.mean())
    std_proc = float(frame_proc.std())
    # SNR for backlit bubble column: fg_mean / bg_std via Otsu split
    # Standard mean/std formula gives negative dB (bubbles raise std).
    # Correct formula: SNR = 20*log10(fg_mean / bg_std).
    _ov, _ = cv2.threshold(frame_proc, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _bg_px = frame_proc[frame_proc <= _ov].astype(float)
    _fg_px = frame_proc[frame_proc >  _ov].astype(float)
    if len(_fg_px) > 10 and len(_bg_px) > 10:
        snr_db = 20.0 * np.log10(_fg_px.mean() / max(_bg_px.std(), 1e-6))
    else:
        snr_db = 0.0
    fg_fraction = float((fg_mask > 0).mean())
    sat_rate = saturation_rate(frame_orig)
    blur_lap = blur_score_laplacian(frame_orig)
    loc_contrast = local_contrast(frame_proc)

    flag = 'ok'
    if snr_db < snr_floor:
        flag = 'low_snr'
    elif sat_rate > 0.25:
        flag = 'saturated'
    elif fg_fraction < 1e-5:
        flag = 'empty'
    elif blur_lap < 1.0:
        flag = 'blurred'

    return {
        'frame_id': int(frame_id),
        'chunk_id': int(chunk_id),
        'mean_raw': mean_raw,
        'std_raw': std_raw,
        'mean_proc': mean_proc,
        'std_proc': std_proc,
        'snr_db': float(snr_db),
        'fg_fraction': fg_fraction,
        'blur_laplacian_var': blur_lap,
        'saturation_rate': sat_rate,
        'local_contrast': loc_contrast,
        'quality_flag': flag,
    }

print('log_frame_quality()  defined  (Otsu SNR + blur + saturation + local contrast)')

---
## Cell 5 — Visual Before / After Check on a Single Frame
Validate the full preprocessing chain on one representative frame before batch processing.

In [ ]:
# Pick a mid-video frame with active bubbles for the inspection
# Pick a mid-video frame with active bubbles for the inspection
INSPECT_FRAME_IDX = 100

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, INSPECT_FRAME_IDX)
ret, raw_frame = cap.read()
cap.release()



# Step 1: Crop to ROI
crop_bgr, sx, sy = crop_to_roi(raw_frame, ROI, scale=1.0)  # keep full-res for inspection
crop_gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

# Step 2: MOG2 on a short sequence around this frame
cap2 = cv2.VideoCapture(str(video_path))
seq_frames = []
for idx in range(max(0, INSPECT_FRAME_IDX - 20), INSPECT_FRAME_IDX + 1):
    cap2.set(cv2.CAP_PROP_POS_FRAMES, idx)
    r, f = cap2.read()
    if r:
        g, _, _ = crop_to_roi(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), ROI, scale=1.0)
        seq_frames.append(g)
cap2.release()

fg_masks_seq, bg_model = background_subtraction_mog2(
    seq_frames, MOG2_HISTORY, MOG2_VAR_THRESHOLD, MOG2_DETECT_SHADOWS
)
fg_mask_inspect = fg_masks_seq[-1]

# Step 3: CLAHE contrast normalisation
clahe_frame = normalize_contrast(crop_gray, CLAHE_CLIP_LIMIT, CLAHE_TILE_GRID)

# Step 4: Bilateral denoising
denoised_frame = denoise_bilateral(clahe_frame, BILATERAL_D,
                                   BILATERAL_SIGMA_C, BILATERAL_SIGMA_S)

# Step 5: Optional adaptive threshold
if USE_ADAPTIVE_THRESHOLD:
    adapt_mask = adaptive_threshold(denoised_frame, ADAPT_BLOCK_SIZE, ADAPT_C)
else:
    adapt_mask = None

# Quality log for this frame
qlog = log_frame_quality(crop_gray, denoised_frame, fg_mask_inspect,
                          INSPECT_FRAME_IDX, chunk_id=-1, snr_floor=SNR_FLOOR_DB)
print(f"Quality log for frame {INSPECT_FRAME_IDX}:")
for k, v in qlog.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# ── Before / After visualisation ─────────────────────────────────────────────
# Because the ROI is tall (3570 px × 360 px), we show horizontal strips
# at three heights: top (interface), mid (column body), bottom (sparger zone)

h_roi = denoised_frame.shape[0]
w_roi = denoised_frame.shape[1]

STRIPS = {
    'Top (interface)':     (0,            min(600, h_roi // 3)),
    'Middle (column body)':(h_roi // 3,   2 * h_roi // 3),
    'Bottom (sparger zone)':(2 * h_roi // 3, h_roi),
}

n_strips = len(STRIPS)
fig, axes = plt.subplots(n_strips, 4, figsize=(18, 4 * n_strips))

for row_idx, (strip_name, (y0s, y1s)) in enumerate(STRIPS.items()):
    raw_s    = crop_gray[y0s:y1s, :]
    clahe_s  = clahe_frame[y0s:y1s, :]
    denoised_s = denoised_frame[y0s:y1s, :]
    fg_s     = fg_mask_inspect[y0s:y1s, :]

    for col_idx, (title, img, cmap) in enumerate([
        ('Raw crop',       raw_s,     'hot'),
        ('After CLAHE',    clahe_s,   'hot'),
        ('After bilateral',denoised_s,'hot'),
        ('MOG2 fg mask',   fg_s,      'gray'),
    ]):
        ax = axes[row_idx, col_idx]
        ax.imshow(img, cmap=cmap, vmin=0, vmax=255, aspect='auto')
        if row_idx == 0:
            ax.set_title(title, fontsize=11, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(strip_name, fontsize=9, rotation=90, va='center')
        ax.axis('off')

plt.suptitle(
    f'NB-01 · Before / After Preprocessing · Frame {INSPECT_FRAME_IDX}',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(OUT / 'before_after_frame.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved → {OUT / 'before_after_frame.png'}")

---
## Cell 6 — Intensity Histogram Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, (label, img) in zip(axes, [
    ('Raw crop',         crop_gray),
    ('After CLAHE',      clahe_frame),
    ('After bilateral',  denoised_frame),
]):
    hist, bins = np.histogram(img.ravel(), bins=128, range=(0, 255))
    ax.fill_between(bins[:-1], hist, alpha=0.7, color='orangered')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Intensity', fontsize=9)
    ax.set_ylabel('Pixel count', fontsize=9)
    ax.set_xlim(0, 255)
    stats_str = f"μ={img.mean():.1f}  σ={img.std():.1f}\nmax={img.max()}  min={img.min()}"
    ax.text(0.97, 0.97, stats_str, transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle(
    f'Intensity Histograms · ROI crop · Frame {INSPECT_FRAME_IDX}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUT / 'histograms_frame.png', dpi=120, bbox_inches='tight')
plt.show()
print("Histograms saved.")
print("\n[GATE CHECK] Does the processed histogram show improved spread / bubble signal?")
print(f"  Raw mean={crop_gray.mean():.1f}  →  Processed mean={denoised_frame.mean():.1f}")
print(f"  Raw std={crop_gray.std():.1f}   →  Processed std={denoised_frame.std():.1f}")

---
## Cell 8 — ROI Overlay Check
Confirm the ROI rectangle and reference lines align with the actual column before batch run.

In [ ]:
# ── up6 ROI overlay check — confirm the cropped region before trusting outputs ──
cap = cv2.VideoCapture(str(video_path))
cap.set(cv2.CAP_PROP_POS_FRAMES, INSPECT_FRAME_IDX)
ret, f = cap.read()
cap.release()
if not ret or f is None:
    raise RuntimeError(f'Could not read frame {INSPECT_FRAME_IDX}')

x0, y0, x1, y1 = ROI
overlay = f.copy()
cv2.rectangle(overlay, (x0, y0), (x1, y1), (0, 255, 0), 5)
for y in REF_LINES:
    cv2.line(overlay, (x0, y), (x1, y), (255, 255, 0), 3)

plt.figure(figsize=(6, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title('ROI overlay — frame ' + str(INSPECT_FRAME_IDX))
plt.axis('off')
plt.show()

print('ROI:', ROI)
print('Reference lines:', REF_LINES)
print('This check should show only the useful bubble plume, not the full frame.')

---
## Cell 7 — Batch Processing: All Chunks

In [ ]:
def preprocess_chunk(
    cap: cv2.VideoCapture,
    chunk_id: int,
    start_frame: int,
    end_frame: int,
    roi: list,
    scale: float,
    mog2_history: int,
    mog2_var_thresh: float,
    mog2_shadows: bool,
    clahe_clip: float,
    clahe_grid: tuple,
    bilateral_d: int,
    bilateral_sc: float,
    bilateral_ss: float,
    use_adapt: bool,
    adapt_block: int,
    adapt_c: int,
    snr_floor: float,
    out_dir: Path,
    out_fmt: str,
) -> list:
    """
    Process one chunk: read frames, apply full preprocessing pipeline,
    save outputs, return quality log records.

    Each processed frame is saved as:
        chunk_{cid:03d}/frame_{frame_id:06d}.{fmt}

    The foreground mask is saved alongside:
        chunk_{cid:03d}/mask_{frame_id:06d}.{fmt}

    Returns
    -------
    list of quality-log dicts (one per frame)
    """
    chunk_dir = out_dir / f'chunk_{chunk_id:03d}'
    chunk_dir.mkdir(parents=True, exist_ok=True)

    # Collect raw grayscale crops for this chunk
    raw_crops = []
    frame_ids = []
    for fid in range(start_frame, end_frame):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
        ret, frame = cap.read()
        if not ret:
            continue
        crop_bgr, sx, sy = crop_to_roi(frame, roi, scale=scale)
        gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
        raw_crops.append(gray)
        frame_ids.append(fid)

    if not raw_crops:
        return []

    # up6: save a few raw ROI samples for audit without storing every raw frame.
    raw_sample_dir = chunk_dir / 'raw_roi_samples'
    raw_sample_dir.mkdir(exist_ok=True)
    sample_positions = sorted(set([0, len(raw_crops)//2, len(raw_crops)-1]))
    for pos in sample_positions:
        raw_name = raw_sample_dir / f'raw_roi_frame_{frame_ids[pos]:06d}.png'
        cv2.imwrite(str(raw_name), raw_crops[pos])

    # MOG2 on the full chunk
    fg_masks, bg_model = background_subtraction_mog2(
        raw_crops, mog2_history, mog2_var_thresh, mog2_shadows
    )

    # Save background model for this chunk
    cv2.imwrite(str(chunk_dir / 'bg_model.png'), bg_model)

    quality_log = []
    for i, (raw_gray, fg_mask, fid) in enumerate(zip(raw_crops, fg_masks, frame_ids)):

        # CLAHE
        clahe_out = normalize_contrast(raw_gray, clahe_clip, clahe_grid)

        # Bilateral
        denoised  = denoise_bilateral(clahe_out, bilateral_d, bilateral_sc, bilateral_ss)

        # Optional adaptive threshold
        if use_adapt:
            adapt_out = adaptive_threshold(denoised, adapt_block, adapt_c)
            # Combine: fg_mask OR adapt_out
            combined_mask = cv2.bitwise_or(fg_mask, adapt_out)
        else:
            combined_mask = fg_mask

        # Save processed frame
        fname = f'frame_{fid:06d}.{out_fmt}'
        if out_fmt == 'npy':
            np.save(str(chunk_dir / fname), denoised)
        else:
            cv2.imwrite(str(chunk_dir / fname), denoised)

        # Save foreground mask
        mname = f'mask_{fid:06d}.{out_fmt}'
        if out_fmt == 'npy':
            np.save(str(chunk_dir / mname), combined_mask)
        else:
            cv2.imwrite(str(chunk_dir / mname), combined_mask)

        # Quality record
        qrec = log_frame_quality(raw_gray, denoised, combined_mask,
                                  fid, chunk_id, snr_floor)
        quality_log.append(qrec)

    # Chunk-level scale metadata
    meta = {
        'chunk_id':      chunk_id,
        'start_frame':   start_frame,
        'end_frame':     end_frame,
        'n_frames':      len(frame_ids),
        'roi':           roi,
        'inference_scale': scale,
        'scale_x_inv':   1.0 / scale,   # multiply back to get original-px coords
        'scale_y_inv':   1.0 / scale,
        'roi_x0':        roi[0],
        'roi_y0':        roi[1],
        'processed_at':  datetime.now(timezone.utc).isoformat(),
    }
    with open(chunk_dir / 'chunk_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    return quality_log


print('preprocess_chunk()  defined.')

---
## Cell 8 — Save Quality Log

In [ ]:
# ── RUN ALL CHUNKS ────────────────────────────────────────────────────────────
print(f"Processing {len(CHUNKS)} chunks  |  ROI={ROI}  |  scale={INFERENCE_SCALE}×")
print()

all_quality_log = []
cap_main = cv2.VideoCapture(str(video_path))

for cid, start, end in tqdm(CHUNKS, desc='Chunks', unit='chunk'):
    chunk_log = preprocess_chunk(
        cap=cap_main, chunk_id=cid,
        start_frame=start, end_frame=end,
        roi=ROI, scale=INFERENCE_SCALE,
        mog2_history=MOG2_HISTORY, mog2_var_thresh=MOG2_VAR_THRESHOLD,
        mog2_shadows=MOG2_DETECT_SHADOWS,
        clahe_clip=CLAHE_CLIP_LIMIT, clahe_grid=CLAHE_TILE_GRID,
        bilateral_d=BILATERAL_D, bilateral_sc=BILATERAL_SIGMA_C,
        bilateral_ss=BILATERAL_SIGMA_S,
        use_adapt=USE_ADAPTIVE_THRESHOLD,
        adapt_block=ADAPT_BLOCK_SIZE, adapt_c=ADAPT_C,
        snr_floor=SNR_FLOOR_DB,
        out_dir=OUT, out_fmt=OUTPUT_FORMAT,
    )
    all_quality_log.extend(chunk_log)
    n_ok  = sum(1 for r in chunk_log if r['quality_flag'] == 'ok')
    n_bad = len(chunk_log) - n_ok
    print(f"  Chunk {cid:2d}: {len(chunk_log):3d} frames  ok={n_ok}  flagged={n_bad}")

cap_main.release()
print(f"\nTotal frames processed: {len(all_quality_log)}")

---
## Cell 9 — Per-Chunk Histogram Dashboard

In [ ]:
df_quality = pd.DataFrame(all_quality_log)

# Deduplicate — keep best SNR for each unique frame_id
# (frames appear in multiple chunks due to overlap)
df_quality_unique = (
    df_quality
    .sort_values('snr_db', ascending=False)
    .drop_duplicates('frame_id')
    .reset_index(drop=True)
)
n_unique = len(df_quality_unique)
print(f'Unique frames after dedup: {n_unique}  (from {len(df_quality)} total with overlap)')
df_quality = df_quality_unique

# Save as Parquet
qlog_parquet = OUTPUT_DIR + 'frame_quality_log.parquet'
df_quality.to_parquet(qlog_parquet, index=False)
# ... rest of Cell 9 unchanged

# Also save as CSV for quick inspection in any tool
qlog_csv = OUTPUT_DIR + 'frame_quality_log.csv'
df_quality.to_csv(qlog_csv, index=False)

print(f"Quality log saved:")
print(f"  Parquet: {Path(qlog_parquet).resolve()}")
print(f"  CSV    : {Path(qlog_csv).resolve()}")
print()
print(df_quality['quality_flag'].value_counts().to_string())
print()
print(df_quality[['snr_db','fg_fraction','mean_raw','mean_proc']].describe().round(3).to_string())

---
## Cell 10 — Visual Spot-Check: One Frame Per Chunk

In [ ]:
def plot_chunk_histograms(
    chunks: list,
    df_quality: pd.DataFrame,
    out_dir: Path,
) -> None:
    """
    Plot per-chunk intensity statistics and fg_fraction over time.
    This satisfies the roadmap acceptance criterion:
    'intensity histogram per chunk confirms signal improvement'.
    """
    n = len(chunks)
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

    frame_ids = df_quality['frame_id'].values

    # Panel 1: mean intensity raw vs processed
    axes[0].plot(frame_ids, df_quality['mean_raw'],  lw=1.0, alpha=0.7,
                 color='orangered', label='Raw mean')
    axes[0].plot(frame_ids, df_quality['mean_proc'], lw=1.0, alpha=0.9,
                 color='limegreen', label='Processed mean')
    axes[0].set_ylabel('Mean intensity', fontsize=10)
    axes[0].legend(fontsize=9, loc='upper right')
    axes[0].set_title('Mean Intensity — Raw vs Processed', fontsize=11)
    axes[0].grid(alpha=0.3)

    # Panel 2: SNR
    axes[1].plot(frame_ids, df_quality['snr_db'], lw=1.0, color='steelblue')
    axes[1].axhline(SNR_FLOOR_DB, ls='--', color='red', lw=1.0,
                    label=f'SNR floor {SNR_FLOOR_DB} dB')
    # Highlight low-SNR frames
    low_snr = df_quality[df_quality['quality_flag'] != 'ok']
    axes[1].scatter(low_snr['frame_id'], low_snr['snr_db'],
                    c='red', s=15, zorder=5, label='Flagged frames')
    axes[1].set_ylabel('SNR (dB)', fontsize=10)
    axes[1].legend(fontsize=9, loc='upper right')
    axes[1].set_title('Signal-to-Noise Ratio per Frame', fontsize=11)
    axes[1].grid(alpha=0.3)

    # Panel 3: fg fraction (bubble occupancy proxy)
    axes[2].plot(frame_ids, df_quality['fg_fraction'] * 100,
                 lw=1.0, color='gold')
    axes[2].set_ylabel('FG fraction (%)', fontsize=10)
    axes[2].set_xlabel('Frame index', fontsize=10)
    axes[2].set_title('Foreground (Bubble) Area Fraction per Frame', fontsize=11)
    axes[2].grid(alpha=0.3)

    # Shade chunk boundaries
    for ax in axes:
        for i, (cid, s, e) in enumerate(chunks):
            if i % 2 == 1:
                ax.axvspan(s, e, alpha=0.06, color='white')
            ax.axvline(s, color='gray', lw=0.5, alpha=0.4)
        ax.axvline(chunks[-1][2], color='gray', lw=0.5, alpha=0.4)

    plt.suptitle('NB-01 · Per-Frame Quality Dashboard', fontsize=13,
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    p = out_dir / 'quality_dashboard.png'
    plt.savefig(p, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"Dashboard saved → {p.resolve()}")


plot_chunk_histograms(CHUNKS, df_quality, OUT)

---
## Cell 11 — Acceptance Gate & Stage Summary

In [ ]:
def spot_check_chunks(chunks: list, out_dir: Path, out_fmt: str) -> None:
    """
    For each chunk, load the middle frame (processed + mask) and display
    a side-by-side strip.  Provides the visual confirmation required by
    the roadmap acceptance criterion.
    """
    n = len(chunks)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]  # normalise shape

    for row, (cid, start, end) in enumerate(chunks):
        mid_fid = (start + end) // 2
        chunk_dir = out_dir / f'chunk_{cid:03d}'

        proc_p = chunk_dir / f'frame_{mid_fid:06d}.{out_fmt}'
        mask_p = chunk_dir / f'mask_{mid_fid:06d}.{out_fmt}'
        bg_p   = chunk_dir / 'bg_model.png'

        # Load or substitute placeholder
        def _load(p, is_npy=False):
            if not p.exists():
                return np.zeros((100, 50), dtype=np.uint8)
            if is_npy or p.suffix == '.npy':
                return np.load(str(p))
            return cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)

        proc_img = _load(proc_p, out_fmt == 'npy')
        mask_img = _load(mask_p, out_fmt == 'npy')
        bg_img   = _load(bg_p)

        # Show a horizontal centre strip for readability (column is very tall)
        h = proc_img.shape[0]
        s0, s1 = h // 3, 2 * h // 3

        for col, (title, img, cmap) in enumerate([
            (f'Chunk {cid} · frame {mid_fid}\nProcessed', proc_img[s0:s1], 'hot'),
            ('MOG2 mask',                                  mask_img[s0:s1], 'gray'),
            ('BG model',                                   bg_img[s0:s1],   'hot'),
        ]):
            ax = axes[row, col]
            ax.imshow(img, cmap=cmap, aspect='auto', vmin=0, vmax=255)
            ax.set_title(title, fontsize=9)
            ax.axis('off')

    plt.suptitle('NB-01 · Spot-Check: Middle Frame per Chunk',
                 fontsize=12, fontweight='bold', y=1.0)
    plt.tight_layout()
    p = out_dir / 'spot_check.png'
    plt.savefig(p, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Spot-check saved → {p.resolve()}")


spot_check_chunks(CHUNKS, OUT, OUTPUT_FORMAT)

---
## Cell 12 — Summary for Gate Documentation

In [ ]:
# ── Quantitative acceptance gate ─────────────────────────────────────────────
n_total = len(df_quality)
n_ok = int((df_quality['quality_flag'] == 'ok').sum())
n_flagged = int(n_total - n_ok)
pct_ok = n_ok / n_total * 100
bad_fraction = n_flagged / n_total

mean_snr = float(df_quality['snr_db'].mean())
mean_fg = float(df_quality['fg_fraction'].mean() * 100)
mean_blur = float(df_quality['blur_laplacian_var'].mean())
mean_saturation = float(df_quality['saturation_rate'].mean() * 100)
mean_local_contrast = float(df_quality['local_contrast'].mean())

gate_pass = bad_fraction <= MAX_BAD_FRAME_FRACTION

print('═' * 60)
print('NB-01  ·  STAGE 1 ACCEPTANCE GATE')
print('═' * 60)
print(f'  Total frames processed   : {n_total}')
print(f'  Quality OK               : {n_ok}  ({pct_ok:.1f} %)')
print(f'  Flagged                  : {n_flagged}  ({bad_fraction*100:.1f} %)')
print(f'  Mean SNR                 : {mean_snr:.2f} dB')
print(f'  Mean FG fraction         : {mean_fg:.3f} %')
print(f'  Mean blur score          : {mean_blur:.2f}')
print(f'  Mean saturation rate     : {mean_saturation:.2f} %')
print(f'  Mean local contrast      : {mean_local_contrast:.2f}')
print(f'  Gate threshold           : <= {MAX_BAD_FRAME_FRACTION*100:.1f} % bad frames')
print(f"  Gate result              : {'PASS - NB-02 authorised' if gate_pass else 'FAIL - review flagged frames'}")
print('═' * 60)

CHUNK_MANIFEST_PATH = OUT / 'chunk_manifest.json'
gate_record = {
    'notebook': 'NB-01',
    'stage': 'Stage 1 — Preprocessing',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'schema_version': '1.0',
    'setup_id': 'setup_A',
    'run_label': RUN_LABEL,
    'n_total': int(n_total),
    'n_ok': int(n_ok),
    'pct_ok': round(float(pct_ok), 3),
    'n_flagged': int(n_flagged),
    'bad_frame_fraction': round(float(bad_fraction), 5),
    'max_bad_frame_fraction': MAX_BAD_FRAME_FRACTION,
    'mean_snr_db': round(mean_snr, 3),
    'mean_fg_frac_pct': round(mean_fg, 5),
    'mean_blur_laplacian_var': round(mean_blur, 3),
    'mean_saturation_rate_pct': round(mean_saturation, 3),
    'mean_local_contrast': round(mean_local_contrast, 3),
    'gate_pass': bool(gate_pass),
    'chunks': [(int(c), int(s), int(e)) for c, s, e in CHUNKS],
    'chunk_manifest_path': str(CHUNK_MANIFEST_PATH),
    'roi': ROI,
    'reference_lines_px': REF_LINES,
    'inference_scale': INFERENCE_SCALE,
    'output_format': OUTPUT_FORMAT,
    'output_dir': str(OUT.resolve()),
}
with open(OUT / 'preprocessing_gate.json', 'w', encoding='utf-8') as f:
    json.dump(gate_record, f, indent=2)

print(f"\nGate record saved -> {(OUT / 'preprocessing_gate.json').resolve()}")

---
## Cell 12 — Summary for Gate Documentation

In [ ]:
from IPython.display import Markdown, display

flag_counts = df_quality['quality_flag'].value_counts().to_dict()

report_md = f"""
## NB-01 Preprocessing Gate Report

| Field | Value |
|---|---|
| **Video** | {video_path.name} |
| **Run label** | {RUN_LABEL} |
| **Resolution** | {V_W} x {V_H} px |
| **FPS (effective)** | {EFFECTIVE_FPS:.4f} |
| **Total frames** | {V_FRAMES} |
| **ROI** | {ROI} |
| **Reference lines** | {REF_LINES} |
| **Inference scale** | {INFERENCE_SCALE}x |
| **Chunks** | {len(CHUNKS)} (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}) |
| **BG subtraction** | MOG2 (history={MOG2_HISTORY}, varThresh={MOG2_VAR_THRESHOLD}) |
| **Contrast norm** | CLAHE (clip={CLAHE_CLIP_LIMIT}, grid={CLAHE_TILE_GRID}) |
| **Denoising** | Bilateral (d={BILATERAL_D}, sigmaC={BILATERAL_SIGMA_C}, sigmaS={BILATERAL_SIGMA_S}) |
| **Adaptive threshold** | {'Enabled' if USE_ADAPTIVE_THRESHOLD else 'Disabled'} |
| **Frames OK / Total** | {n_ok} / {n_total}  ({pct_ok:.1f} %) |
| **Mean SNR** | {mean_snr:.2f} dB |
| **Mean FG fraction** | {mean_fg:.3f} % |
| **Mean blur score** | {mean_blur:.2f} |
| **Mean saturation rate** | {mean_saturation:.2f} % |
| **Mean local contrast** | {mean_local_contrast:.2f} |
| **Flag breakdown** | {flag_counts} |
| **Gate <= {MAX_BAD_FRAME_FRACTION*100:.1f}% bad frames** | {'PASS' if gate_pass else 'FAIL'} |
| **Chunk manifest** | `{CHUNK_MANIFEST_PATH}` |

**Outputs written to:** `{OUT.resolve()}`

**Decision:** {'NB-02 segmentation is authorised to run.' if gate_pass else 'Gate failed — review flagged frames and retune ROI/MOG2/bilateral parameters.'}
"""
display(Markdown(report_md))